# 📚 Import libraries

In [24]:
import hashlib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from collections import defaultdict
import itertools
from threading import Thread
import os # For API keys
import warnings # To suppress warnings
import matplotlib.pyplot as plt
import spacy
from scipy.stats import norm


# 🗂️ Embedding model

## ⚙ Config

In [25]:
# --- Embedding Model Configuration ---
# Choose your embedding provider and model here
EMBEDDING_CONFIG = {
    # Options: "spacy", "sentence-transformers", "huggingface", "openai"
    "provider": "huggingface",
    #"provider": "sentence-transformers",

    # --- Provider Specific Settings ---
    # spacy: Model name (e.g., "en_core_web_sm", "en_core_web_md", "en_core_web_lg")
    "spacy_model_name": "en_core_web_sm",

    # sentence-transformers: Model name from Hugging Face Hub (e.g., "all-MiniLM-L6-v2", "multi-qa-mpnet-base-dot-v1")
    "st_model_name": "all-MiniLM-L6-v2",

    # huggingface: Model name from Hugging Face Hub (e.g., "bert-base-uncased", "roberta-base")
    "hf_model_name": "bert-base-uncased",
    "hf_pooling": "mean", # Options: "mean", "cls"

    # openai: Model name (e.g., "text-embedding-ada-002", "text-embedding-3-small")
    "openai_model_name": "text-embedding-ada-002",
}

# --- Global variables to hold loaded models (lazy loading) ---
_nlp = None
_st_model = None
_hf_tokenizer = None
_hf_model = None
_openai_client = None

# --- Model Loading and Embedding Function ---

def initialize_embedding_model():
    """Loads the embedding model based on EMBEDDING_CONFIG."""
    global _nlp, _st_model, _hf_tokenizer, _hf_model, _openai_client
    provider = EMBEDDING_CONFIG["provider"]

    if provider == "spacy":
        if _nlp is None:
            model_name = EMBEDDING_CONFIG["spacy_model_name"]
            try:
                _nlp = spacy.load(model_name)
                print(f"Loaded spaCy model: {model_name}")
            except OSError:
                print(f"SpaCy model '{model_name}' not found. Please install it: python -m spacy download {model_name}")
                raise
    elif provider == "sentence-transformers":
        if _st_model is None:
            try:
                from sentence_transformers import SentenceTransformer
                model_name = EMBEDDING_CONFIG["st_model_name"]
                _st_model = SentenceTransformer(model_name)
                print(f"Loaded Sentence Transformer model: {model_name}")
            except ImportError:
                print("Sentence Transformers library not found. Please install it: pip install sentence-transformers")
                raise
    elif provider == "huggingface":
        if _hf_model is None or _hf_tokenizer is None:
            try:
                # Suppress warnings about weights not being used for sequence classification
                warnings.filterwarnings("ignore", category=UserWarning, module='transformers.modeling_utils')
                from transformers import AutoTokenizer, AutoModel
                import torch
                warnings.filterwarnings("default", category=UserWarning, module='transformers.modeling_utils') # Reset warning

                model_name = EMBEDDING_CONFIG["hf_model_name"]
                _hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
                _hf_model = AutoModel.from_pretrained(model_name)
                # Use GPU if available
                if torch.cuda.is_available():
                   _hf_model.to('cuda')
                print(f"Loaded Hugging Face model and tokenizer: {model_name}")
            except ImportError:
                print("Hugging Face Transformers library or PyTorch not found. Please install them: pip install transformers torch")
                raise
    elif provider == "openai":
        if _openai_client is None:
            try:
                from openai import OpenAI
                # Requires OPENAI_API_KEY environment variable to be set
                if not os.getenv("OPENAI_API_KEY"):
                    print("Warning: OPENAI_API_KEY environment variable not set for OpenAI embeddings.")
                    # Optionally raise an error here if you require OpenAI
                    # raise ValueError("OPENAI_API_KEY environment variable not set.")
                _openai_client = OpenAI() # Client automatically picks up the key from env var
                print(f"Initialized OpenAI client for model: {EMBEDDING_CONFIG['openai_model_name']}")
            except ImportError:
                print("OpenAI library not found. Please install it: pip install openai")
                raise
    else:
        raise ValueError(f"Unsupported embedding provider: {provider}")

## 🧰 Embedding tools

In [26]:
def get_embedding(text):
    """Generates an embedding for the given text using the configured model."""
    initialize_embedding_model() # Ensure model is loaded
    provider = EMBEDDING_CONFIG["provider"]

    if not text or not text.strip(): # Handle empty or whitespace-only strings
      warnings.warn("Attempting to get embedding for empty or whitespace-only text. Returning zero vector.")
      # Return a zero vector of the expected dimension. This is tricky without knowing the model first.
      # Let's try to get the dimension from the loaded model if possible, otherwise default.
      dim = 300 # Default fallback dimension
      if provider == "spacy" and _nlp and _nlp.vocab.vectors_length > 0:
          dim = _nlp.vocab.vectors_length
      elif provider == "sentence-transformers" and _st_model:
          dim = _st_model.get_sentence_embedding_dimension()
      elif provider == "huggingface" and _hf_model:
          dim = _hf_model.config.hidden_size
      elif provider == "openai":
           # OpenAI dims vary by model, ada-002 is 1536, text-embedding-3-small is 1536
           if EMBEDDING_CONFIG["openai_model_name"] in ["text-embedding-ada-002", "text-embedding-3-small"]:
               dim = 1536
           elif EMBEDDING_CONFIG["openai_model_name"] == "text-embedding-3-large":
               dim = 3072
           # Add other OpenAI models or a lookup mechanism if needed
      return np.zeros(dim)


    if provider == "spacy":
        doc = _nlp(text)
        return doc.vector
    elif provider == "sentence-transformers":
        # The encode function expects a list of sentences, but works fine with a single string too
        embedding = _st_model.encode(text)
        return embedding # Already a numpy array
    elif provider == "huggingface":
        import torch
        inputs = _hf_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        # Move inputs to GPU if model is on GPU
        if torch.cuda.is_available():
            inputs = {k: v.to('cuda') for k, v in inputs.items()}

        with torch.no_grad():
            outputs = _hf_model(**inputs)

        # Pooling strategy
        if EMBEDDING_CONFIG["hf_pooling"] == "mean":
            # Mean pooling: Average token embeddings, ignore padding
            attention_mask = inputs['attention_mask']
            mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            embedding = sum_embeddings / sum_mask
        elif EMBEDDING_CONFIG["hf_pooling"] == "cls":
            # CLS pooling: Use the embedding of the [CLS] token
            embedding = outputs.last_hidden_state[:, 0, :]
        else:
            raise ValueError(f"Unsupported Hugging Face pooling strategy: {EMBEDDING_CONFIG['hf_pooling']}")

        # Move embedding to CPU and convert to numpy
        return embedding.cpu().numpy().squeeze()

    elif provider == "openai":
        if not _openai_client:
             # Handle case where API key might be missing and client wasn't initialized
             print("Error: OpenAI client not available. Cannot get embedding.")
             # Return zero vector or raise error, similar to empty text handling
             # Using the dimension logic from above
             dim = 1536 # Default for common models
             if EMBEDDING_CONFIG["openai_model_name"] == "text-embedding-3-large":
                 dim = 3072
             return np.zeros(dim)

        response = _openai_client.embeddings.create(
            input=text,
            model=EMBEDDING_CONFIG["openai_model_name"]
        )
        return np.array(response.data[0].embedding)

    else:
        # Should have been caught by initialize_embedding_model, but just in case
        raise ValueError(f"Unsupported embedding provider: {provider}")


###
def calculate_distance(embedding1, embedding2):
    """Calculates the cosine distance between two embeddings."""
    # Ensure inputs are numpy arrays
    embedding1 = np.asarray(embedding1)
    embedding2 = np.asarray(embedding2)

    # Check for zero vectors
    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)
    if norm1 == 0 or norm2 == 0:
        if norm1 == 0 and norm2 == 0:
            return 0.0 # Distance between two zero vectors is 0
        else:
            return 1.0 # Distance between zero vector and non-zero vector is 1 (max cosine distance)

    # Reshape for cosine_similarity function (expects 2D arrays)
    if embedding1.ndim == 1:
        embedding1 = embedding1.reshape(1, -1)
    if embedding2.ndim == 1:
        embedding2 = embedding2.reshape(1, -1)

    # Calculate cosine similarity and convert to distance
    similarity = cosine_similarity(embedding1, embedding2)[0][0]
    # Clip similarity to [-1, 1] range due to potential floating point inaccuracies
    similarity = np.clip(similarity, -1.0, 1.0)
    return 1 - similarity

# 📊 Plotting configs

In [27]:
# Default threshold configuration with multiple levels
THRESHOLD_CONFIG = {
    "context": {
        "method": "gaussian_average",  # Options: "gaussian_average", "custom", None
        "custom_value": 0.1,           # Value to use if method is "custom"
        "levels": [ 0.0,0.5, 0.75, 0.9],     # Standard deviations above the mean for levels
        "colors": ['lightcoral', 'salmon', 'red', "yellow"] # Corresponding colors for levels
    },
    "query": {
        "method": "gaussian_average",  # Options: "gaussian_average", "custom", None
        "custom_value": 0.1,           # Value to use if method is "custom"
        "levels": [0.0,0.5, 0.75],     # Standard deviations above the mean for levels
        "colors": ['lightblue', 'cornflowerblue', 'blue', 'black'] # Corresponding colors for levels
    },
}

# 🧮 Define Classes & Functions

## 🧰 Classes

In [28]:
class RagThread(Thread):
    def __init__(self, group=None, target=None, name=None, args=(), kwargs={}, verbose=None):
        # Initializing the Thread class
        super().__init__(group, target, name, args, kwargs)
        self._return = None

    # Overriding the Thread.run function
    def run(self):
        if self._target is not None:
            self._return = self._target(*self._args, **self._kwargs)

    def join(self):
        super().join()
        return self._return

## ⚛ Functions

###  Tokenizer

In [29]:
# Load spaCy *only* for tokenization if not used for embeddings
_tokenizer_nlp = None
def get_tokenizer_nlp():
    """Loads a small spaCy model specifically for tokenization if needed."""
    global _tokenizer_nlp
    if _tokenizer_nlp is None:
        try:
            # Using small model just for tokenization is efficient
            _tokenizer_nlp = spacy.load("en_core_web_sm")
        except OSError:
            print("Default spaCy model 'en_core_web_sm' not found for tokenization. Please install it: python -m spacy download en_core_web_sm")
            # Fallback to basic split if spacy is unavailable
            print("Warning: Falling back to basic whitespace tokenization.")
            _tokenizer_nlp = 'fallback'
    return _tokenizer_nlp

def tokenize_and_filter(text):
    """Splits text into sentences instead of tokens."""
    tokenizer_nlp = get_tokenizer_nlp()

    if tokenizer_nlp == 'fallback':
        # Basic fallback if spaCy isn't available
        # Simple sentence splitting by punctuation and cleanup
        import re

        # Split on common sentence endings
        sentences = re.split(r'(?<=[.!?])\s+', text)

        # Remove empty sentences and strip whitespace
        sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

        return sentences

    # Use spaCy for more robust sentence tokenization
    doc = tokenizer_nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    return sentences

### Subtitutions

In [30]:
####
def remove_and_compute(sentences, original_response_embedding, category, sentence_impact_dict, original_query):
    """
    Computes the impact of removing each sentence.
    Uses hash of sentences as dictionary keys for consistent lookup.
    """
    impact_values = []
    new_sentence_impact_dict = defaultdict(list)

    # Process each sentence
    for i, sentence_to_remove in enumerate(sentences):
        # Create a hash of the sentence for consistent key generation
        sentence_hash = hashlib.md5(sentence_to_remove.encode()).hexdigest()

        # Create text with the current sentence removed
        remaining_sentences = [s for j, s in enumerate(sentences) if i != j]
        remaining_text = " ".join(remaining_sentences)

        text_to_evaluate = ""
        if remaining_text.strip():
            text_to_evaluate = remaining_text

        text_to_evaluate_embedding = get_embedding(text_to_evaluate)
        distance_change = calculate_distance(text_to_evaluate_embedding, original_response_embedding)

        key = f"{category}_{sentence_hash}"
        if key in new_sentence_impact_dict:
            new_sentence_impact_dict[key].append(distance_change)
        else:
            new_sentence_impact_dict[key] = [distance_change]

        impact_values.append(distance_change)

    # Calculate mean impact across all sentences
    if impact_values:
        mean_impact = np.mean(impact_values)
        std_impact = np.std(impact_values) if np.std(impact_values) > 0 else 1

        max_impact = 1
        # Normalize the impacts relative to the mean
        for key, values in new_sentence_impact_dict.items():
            normalized_impact = (np.mean(values) - mean_impact) / (std_impact)
            if normalized_impact > max_impact:
                max_impact = normalized_impact
            if(key in sentence_impact_dict):
                sentence_impact_dict[key].append(normalized_impact)
            else:
                sentence_impact_dict[key] = [normalized_impact]

        for key, values in sentence_impact_dict.items():
            sentence_impact_dict[key] = [v / max_impact for v in values]  # Normalize by max impact (to make the range in -1, 1)

#####
def explain_by_token_removal(chunk, query, response):
    """
    Explains the influence of each token in the chunk and query
    by measuring the change in embedding distance of the response
    when each token is removed.
    """
    response_embedding = get_embedding(response) # Embedding of the actual RAG response
    query_embedding = get_embedding(query)

    chunk_sentences = tokenize_and_filter(chunk)
    query_sentences = tokenize_and_filter(query)

    token_impact = defaultdict(list)

    print("Analyzing chunk tokens wrt response...")
    remove_and_compute(chunk_sentences, response_embedding, "context", token_impact, query)

    print("Analyzing chunk tokens wrt query...")
    remove_and_compute(chunk_sentences, query_embedding, "context", token_impact, query)

    print("Analyzing query tokens wrt response...")
    remove_and_compute(query_sentences, response_embedding, "query", token_impact, chunk)

    for key in  token_impact.keys():
        token_impact[key] = np.mean(token_impact[key])


    return token_impact



### Top-k explanations per sentece

In [31]:
#####
def rag_simulate(context, query):
    """
    A very basic simulation of a RAG system.
    In a real system, this would involve retrieving relevant parts of the context
    based on the query and then generating an answer.
    For this explainability task, we'll simply concatenate them (with a separator).
    This allows us to observe how changes in context/query affect the embedding
    of this "simulated" response.
    """
    return f"Context: {context} Query: {query}"


#####
def explain_by_sentence(chunk, response,k):
    """
    Explains the influence of each sentence in the chunk by measuring
    the change in embedding distance of the response when each sentence is removed.
    """
    chunk_sentences = tokenize_and_filter(chunk)
    response_sentences = tokenize_and_filter(response)

    sentence_impact = defaultdict(tuple)
    for response_sentence in response_sentences:
        sentence_hash = hashlib.md5(response_sentence.encode()).hexdigest()
        key = f"sentence_{sentence_hash}"
        response_sentence_embedding = get_embedding(response_sentence)

        for chunk_sentence in chunk_sentences:
            # Create a hash of the sentence for consistent key generation
            chunk_sentence_embedding = get_embedding(chunk_sentence)
            distance_change = calculate_distance(chunk_sentence_embedding, response_sentence_embedding)
            tuple_response_chunk = (chunk_sentence, distance_change)
            if key in sentence_impact :
                if len(sentence_impact[key]) < k:
                    sentence_impact[key].append(tuple_response_chunk)
                else:
                    for(s, d) in sentence_impact[key]:
                        if d > distance_change:
                            sentence_impact[key].remove((s, d))
                            sentence_impact[key].append(tuple_response_chunk)
                            break
            else:
                sentence_impact[key] = [tuple_response_chunk]
            sentence_impact[key] = sorted(sentence_impact[key], key=lambda x: x[1])# Keep only top k sentences

    return sentence_impact




### Utils

In [32]:
def explain_by_context_permutation(context, query, response, num_permutations=5):
    """
    Explains the influence of context syntactic formulation by measuring
    the change in embedding distance of the response for different
    permutations of the context sentences.
    """
    response_embedding = get_embedding(response) # Original response embedding
    context_hash = hashlib.md5(context.encode()).hexdigest() # Hash for consistent key generation

    # Use spaCy for sentence splitting
    tokenizer_nlp = get_tokenizer_nlp()
    if tokenizer_nlp == 'fallback':
        print("Warning: Cannot perform sentence permutation analysis without spaCy. Skipping.")
        return {}
    sentences = [sent.text.strip() for sent in tokenizer_nlp(context).sents if sent.text.strip()]

    if len(sentences) < 2:
        print("Context has less than 2 sentences. Skipping permutation analysis.")
        return {}

    permutation_impact = defaultdict(float)
    unique_permutations = set() # To avoid redundant permutations if num_permutations is large


    # Ensure the original order is considered if possible within the limit
    unique_permutations.add(context)

    all_perms = []
    for i in range(5):
        all_perms.append(np.random.permutation(sentences))
    np.random.shuffle(all_perms) # Shuffle to get varied permutations if num_permutations is smaller than total

    count = 0
    for perm_tuple in all_perms:
        permuted_context = " ".join(perm_tuple)
        if permuted_context not in unique_permutations:
            unique_permutations.add(permuted_context)
            count += 1
            if count >= num_permutations: # Stop after reaching the desired number *excluding* the original
                 break

    # If original wasn't added due to limit, remove one random to make space if needed
    if context not in unique_permutations and len(unique_permutations) >= num_permutations:
         unique_permutations.pop() # Remove a random element
         unique_permutations.add(context)


    print(f"Analyzing {len(unique_permutations)} context permutations (including original)...")
    for perm_context in unique_permutations:
        # Simulate RAG with the permuted context
        temp_simulated_response = rag_simulate(perm_context, query)
        temp_response_embedding = get_embedding(temp_simulated_response)

        # Compare the embedding of the *simulated response from permuted context*
        # to the embedding of the *original actual response*
        distance_change = calculate_distance(temp_response_embedding, response_embedding)
        permutation_hash = hashlib.md5(perm_context.encode()).hexdigest() # Hash for consistent key generation
        if permutation_hash == context_hash:
            permutation_impact["original"] = distance_change # Store distance, not change from original perm
        else:
            permutation_impact[perm_context] = distance_change # Store distance, not change from original perm

    return permutation_impact


def calculate_token_thresholds_with_levels(token_impact, threshold_config):
    """Calculates separate thresholds with multiple levels for context and query tokens."""
    context_impacts = [v for k, v in token_impact.items() if k.startswith("context_")]
    query_impacts = [v for k, v in token_impact.items() if k.startswith("query_")]
    thresholds = {}

    def get_thresholds_and_colors(impacts, config):
        method = config.get("method")
        levels = config.get("levels", [])
        colors = config.get("colors", [])
        threshold_colors = []
        if not impacts: # Handle cases with no tokens
            return [] # Return empty list if no impacts to calculate from

        # Ensure impacts are numbers
        impacts = [float(i) for i in impacts if isinstance(i, (int, float, np.number))]
        if not impacts: # Handle case where impacts were not numbers
             return []

        if method == "gaussian_average":
            avg = np.mean(impacts)
            std = np.std(impacts)
            # Ensure std is not zero or extremely small to avoid issues
            if std < 1e-9:
                std = 1e-9 # Use a small epsilon if std dev is effectively zero

            for i, level in enumerate(levels):
                threshold = avg + level
                print(f"Threshold {i}: {threshold:.8f} (Level: {level})")
                color = colors[i] if i < len(colors) else 'gray'
                threshold_colors.append((threshold, color))
            return sorted(threshold_colors) # Sort by threshold value ascending
        elif method == "custom":
            custom_value = config.get("custom_value", 0.0) # Default custom base to 0
            for i, level in enumerate(levels):
                # Assuming levels for custom are additive offsets from custom_value
                threshold = custom_value + level
                color = colors[i] if i < len(colors) else 'gray'
                threshold_colors.append((threshold, color))
            return sorted(threshold_colors) # Sort by threshold value ascending
        elif method is None:
             return [] # No thresholds calculated
        else:
            print(f"Warning: Unknown threshold method '{method}'. No thresholds calculated.")
            return []

    thresholds["context"] = get_thresholds_and_colors(context_impacts, threshold_config.get("context", {}))
    thresholds["query"] = get_thresholds_and_colors(query_impacts, threshold_config.get("query", {}))
    return thresholds


# 📈 Analyze RAG

## 💟 Core function

In [33]:
def analyze_rag(context, query, response, analyze_permutations=True, num_permutations=5):
    """
    Performs a complete RAG analysis and returns the results.

    Args:
        context: The context text for RAG
        query: The query text
        response: The response generated by the RAG system
        analyze_permutations: Whether to analyze context permutations
        num_permutations: Number of permutations to analyze if enabled

    Returns:
        Dictionary containing token_impact, permutation_impact, and thresholds
    """
    print(f"--- Starting RAG Analysis ---")
    print(f"Embedding Provider: {EMBEDDING_CONFIG['provider']}")
    # Print specific model name based on provider
    provider = EMBEDDING_CONFIG['provider']
    model_key = f"{provider.split('-')[0]}_model_name" # e.g. spacy_model_name, st_model_name
    if model_key in EMBEDDING_CONFIG:
      print(f"Embedding Model: {EMBEDDING_CONFIG[model_key]}")
    elif provider == 'huggingface':
        print(f"Embedding Model: {EMBEDDING_CONFIG['hf_model_name']} (Pooling: {EMBEDDING_CONFIG['hf_pooling']})")


    # Initialize the model before analysis starts
    try:
        initialize_embedding_model()
    except Exception as e:
        print(f"Error initializing embedding model: {e}")
        # Return partial results or raise error
        return {
            "error": f"Failed to initialize embedding model: {e}",
            "token_impact": {},
            "permutation_impact": {},
            "thresholds": {},
            "context": context,
            "query": query,
            "response": response,
            "threshold_config": THRESHOLD_CONFIG,
            "embedding_config": EMBEDDING_CONFIG
        }

    token_impact = defaultdict(float)
    #multi-threading for each chunk of context which is an array of chunks
    threads = []
    for i, chunk in enumerate(context):
        print(f"Processing chunk {i+1}/{len(context)}: {chunk[:50]}...")
        thread = RagThread(target=explain_by_token_removal, args=(chunk, query, response))
        thread.start()
        threads.append(thread)

    for i in range(len(context)):
        test = threads[i].join()
        token_impact.update(test)

    # Analyze wich top k sentences in the context chunks
    sentence_impact = defaultdict(float)
    threads = []
    k = 4
    for i, chunk in enumerate(context):
        print(f"Processing chunk {i+1}/{len(context)}: {chunk[:50]}...")
        thread = RagThread(target=explain_by_sentence, args=(chunk, response,k))
        thread.start()
        threads.append(thread)


    for i in range(len(context)):
        test = threads[i].join()
        # results from each thread are merged into the main dictionary with the tuples updated depending on the distance values, every thread returns a dictionary which for each key has an array of tuples with the sentence and the distance value
        for key, value in test.items():
            if key in sentence_impact:
                sentence_impact[key].extend(value)
            else:
                sentence_impact[key] = value


    # sort the dictionary by distance value
    for key in sentence_impact.keys():
        sentence_impact[key] = sorted(sentence_impact[key], key=lambda x: x[1])[:k]



    # Analyze permutations if requested
    permutation_impact = {}
    if analyze_permutations:
        permutation_impact = explain_by_context_permutation(
            context, query, response, num_permutations=num_permutations
        )

    # Calculate thresholds
    thresholds = calculate_token_thresholds_with_levels(token_impact, THRESHOLD_CONFIG)
    print(f"--- Analysis Complete ---")

    return {
        "token_impact": token_impact,
        "permutation_impact": permutation_impact,
        "sentence_impact": sentence_impact,
        "thresholds": thresholds,
        "context": context,
        "query": query,
        "response": response,
        "threshold_config": THRESHOLD_CONFIG,
        "embedding_config": EMBEDDING_CONFIG # Include embedding config in results
    }

## 🔤 Define context/chunks -  query - response

In [34]:
contexts = [
        "The COVID-19 pandemic disrupted nearly every aspect of global trade. Lockdowns led to sudden factory shutdowns in key manufacturing hubs such as China and Vietnam. Shipping containers were stranded in the wrong ports, causing logistical delays. Some companies experienced increased demand for essential goods, while others saw sharp declines. These mismatches made planning difficult and unpredictable.",
        "Companies quickly began looking for alternative sourcing strategies. For example, many firms shifted from just-in-time to just-in-case inventory models to improve resilience. Investments in digital supply chain technologies, like AI forecasting tools, also accelerated. However, small and medium-sized enterprises struggled to keep up due to limited capital and technical expertise.",
        "Global supply chain issues were also impacted by geopolitical tensions unrelated to the pandemic, such as Brexit and the US-China trade war. These factors exacerbated existing fragilities but were not the primary cause of the COVID-specific disruption. Labor shortages due to illness and quarantine also played a significant role.",
        "Environmental factors such as droughts and hurricanes affected shipping routes and agricultural production, contributing to delays and shortages. However, these were secondary compared to the sudden changes in consumer demand and supplier availability triggered by the pandemic."
    ]
query = "What were the main causes of the global supply chain crisis during the COVID-19 pandemic, and how did companies adapt to these disruptions?"
response = "The global supply chain crisis during COVID-19 was driven by factory shutdowns, shipping disruptions, and mismatches in supply and demand. Companies adapted by adopting more resilient inventory strategies and accelerating digital transformation. Labor shortages due to quarantine also worsened the situation."


# 🟥 Main

In [35]:
def run_analysis(context, query, response, analyze_permutations=True, num_permutations=5,
                generate_html=True, generate_plots=True, html_filename="rag_explanation.html"):
    """
    Run a complete RAG explainability analysis

    Args:
        context: Context text used in RAG
        query: Query text used in RAG
        response: Response generated by RAG
        analyze_permutations: Whether to analyze context permutations
        num_permutations: Number of permutations to analyze
        generate_html: Whether to generate HTML visualization
        generate_plots: Whether to generate embedding plots
        html_filename: Output filename for HTML report
    """
    # Run analysis
    print("Running RAG explainability analysis...")
    analysis_results = analyze_rag(
        context, query, response,
        analyze_permutations=analyze_permutations,
        num_permutations=num_permutations
    )
    impact_filter=(0.5, 1)
    # Print token impact results
    print("\nToken Impact:")
    for token, impact in sorted(analysis_results["token_impact"].items(), key=lambda item: item[1], reverse=True):
        print(f"{token}: {impact:.4f}")

    # Print permutation results if available
    if analyze_permutations and analysis_results["permutation_impact"]:
        print("\nContext Permutation Impact:")
        for perm, impact in sorted(analysis_results["permutation_impact"].items(), key=lambda item: item[1], reverse=True):
            print(f"{impact:.4f}: {perm[:50]}...")  # Print first 50 chars

    # Generate HTML if requested
    #if generate_html:
     #   save_html_report(analysis_results, filename=html_filename)

    context_impacts  = {k: v for k,v in analysis_results["token_impact"].items() if k.startswith("context_") }
    def calculate_percentage(value, impact_dict):
        if not impact_dict:
            return 0
        max_value = max(float(v) for v in impact_dict.values())
        max_value = max(max_value, 1)  # Avoid division by zero
        min_value = min(float(v) for v in impact_dict.values())
        min_value = min(min_value, -1)  # Avoid division by zero

        if max_value == 0:
            return 0
        return (float(value)-min_value) / (max_value- min_value)
    filtered_context_impacts = {k:v for k,v in context_impacts.items() if calculate_percentage(v, context_impacts) > impact_filter[0] and calculate_percentage(v, context_impacts) <= impact_filter[1] }
    pseudo_response=""
    for chunk in context:
        chunk_sentences = tokenize_and_filter(chunk)
        for sentence in chunk_sentences:
            hash = f"context_{hashlib.md5(sentence.encode()).hexdigest()}"
            if hash in filtered_context_impacts.keys():
                pseudo_response += sentence

    analysis_results["pseudo_response"] = pseudo_response
    # Generate plots if requested
    #if generate_plots:
       #plot_embeddings(analysis_results, get_embedding, impact_filter=impact_filter)


run_analysis(
        context=contexts,
        query=query,
        response=response,
        analyze_permutations=False,
        num_permutations=5,
        generate_html=True,
        generate_plots=True
    )

Running RAG explainability analysis...
--- Starting RAG Analysis ---
Embedding Provider: huggingface
Embedding Model: bert-base-uncased (Pooling: mean)
Loaded Hugging Face model and tokenizer: bert-base-uncased
Processing chunk 1/4: The COVID-19 pandemic disrupted nearly every aspec...
Processing chunk 2/4: Companies quickly began looking for alternative so...
Processing chunk 3/4: Global supply chain issues were also impacted by g...
Processing chunk 4/4: Environmental factors such as droughts and hurrica...
Analyzing chunk tokens wrt response...
Analyzing chunk tokens wrt response...
Analyzing chunk tokens wrt response...
Analyzing chunk tokens wrt response...
Analyzing chunk tokens wrt query...
Analyzing query tokens wrt response...


<ipython-input-26-106b6035e519>:7: UserWarning: Attempting to get embedding for empty or whitespace-only text. Returning zero vector.
  warnings.warn("Attempting to get embedding for empty or whitespace-only text. Returning zero vector.")


Analyzing chunk tokens wrt query...
Analyzing chunk tokens wrt query...
Analyzing chunk tokens wrt query...
Analyzing query tokens wrt response...


<ipython-input-26-106b6035e519>:7: UserWarning: Attempting to get embedding for empty or whitespace-only text. Returning zero vector.
  warnings.warn("Attempting to get embedding for empty or whitespace-only text. Returning zero vector.")


Analyzing query tokens wrt response...


<ipython-input-26-106b6035e519>:7: UserWarning: Attempting to get embedding for empty or whitespace-only text. Returning zero vector.
  warnings.warn("Attempting to get embedding for empty or whitespace-only text. Returning zero vector.")


Analyzing query tokens wrt response...
Processing chunk 1/4: The COVID-19 pandemic disrupted nearly every aspec...
Processing chunk 2/4: Companies quickly began looking for alternative so...
Processing chunk 3/4: Global supply chain issues were also impacted by g...
Processing chunk 4/4: Environmental factors such as droughts and hurrica...


<ipython-input-26-106b6035e519>:7: UserWarning: Attempting to get embedding for empty or whitespace-only text. Returning zero vector.
  warnings.warn("Attempting to get embedding for empty or whitespace-only text. Returning zero vector.")


Threshold 0: -0.00000007 (Level: 0.0)
Threshold 1: 0.49999993 (Level: 0.5)
Threshold 2: 0.74999993 (Level: 0.75)
Threshold 3: 0.89999993 (Level: 0.9)
Threshold 0: 0.00000000 (Level: 0.0)
Threshold 1: 0.50000000 (Level: 0.5)
Threshold 2: 0.75000000 (Level: 0.75)
--- Analysis Complete ---

Token Impact:
context_0de5f3477ddd5c7cbaa2c92f65616cfc: 1.0000
context_af0b07980e40f316fd784d1d6178b8c4: 0.8848
context_bfdd028c787b0babccb04ce4f90e880b: 0.7280
context_aa73f30eeae1f5544f9577466c269208: 0.6981
context_86b8a3b6547367df723ed02bae1a3bc5: 0.4674
context_b1730346ed2d776c46ab66e8eab6e07c: 0.2075
context_6997bceaf04342a4eacfdd428748cc3c: 0.2049
context_a3d973573f8e57577508bbcef89ac118: 0.0037
query_6251af9b41eda766feeb04f0748d760b: 0.0000
context_cf884df2de0169d0ab15ab7fac7b55b1: -0.2269
context_bbeb3ae39bd961b68f94ff22dafb2ff6: -0.3376
context_52f70713d6c06a61ef2ba97ac8e6fa60: -0.5472
context_2c9d892fb2d2037aa327993ff8407754: -0.6529
context_b1e643fadc382c1e9751640a3d83f564: -1.0000
context_

In [37]:
## Define context/chunks